In [5]:
# imports
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [6]:
# data

#file_path = r"C:\Users\mrang\Downloads\master_0_dataset.csv"
file_path = "../data/master_0_dataset.csv"
data = pd.read_csv(file_path)
data.head()

,id,text,label,generation_type,source,source_id
0,0,Have you ever pondered the evolution of indiv...,ai,zero_shot,ai_essay,ai_essay_21
1,1,"Dear Reader,\n\nLet us embark on an explorati...",ai,zero_shot,ai_essay,ai_essay_9
2,2,London: Crude prices fell Thursday awaiting th...,human,human,articles.csv,articles_324
3,3,ISLAMABAD: Pakistan captain Inamul Haq is opti...,human,human,articles.csv,articles_1592
4,4,Title: A Sunny Day Revival at Culver's After ...,ai,zero_shot,ai_review,ai_review_12


Here is used the 10 fold cross validation, so randomly split the data, and the model picked up on the source and formatting, giving very high accuracy scores.

In [49]:
#Ensure text is string and normalize labels
data['text'] = data['text'].astype(str)
data['label'] = data['label'].str.lower().str.strip()

#Convert labels (numeric)
label_map = {'human': 0, 'ai': 1}
data['label_num'] = data['label'].map(label_map)


#Separate feats (text) and labels
X_text = data['text']
y = data['label_num']


#Convert text into numerical features using TFIDF
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(X_text)

#Naive Bayes
model = MultinomialNB()


# Perform 10-fold cross-validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

accuracy_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Accuracy scores for each fold:")
print(accuracy_scores)

print("\nMean accuracy:", accuracy_scores.mean())

#predictions
y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv
)


#Print classification report
#precision, recall, and F1-score for both classes
print(classification_report(y, y_pred, target_names=['human', 'ai']))

Accuracy scores for each fold:
[0.99166667 0.99166667 1.         1.         0.99166667 0.98333333
 0.99166667 0.975      0.98333333 1.        ]

Mean accuracy: 0.9908333333333333
              precision    recall  f1-score   support

       human       0.99      0.99      0.99       600
          ai       0.99      0.99      0.99       600

    accuracy                           0.99      1200
   macro avg       0.99      0.99      0.99      1200
weighted avg       0.99      0.99      0.99      1200



Here i attempted a different method and used grouped cross validation (GroupKfold) so the data was split by source instead of randomly, removed the noticeable difference in data and this is where the accuracy dropped, so the model was relying on those differences instead of learning the text.

In [50]:
groups = subset['source']

group_cv = GroupKFold(n_splits=2)

y_pred_group = cross_val_predict(
    pipeline,
    subset['text'],
    subset['label_num'],
    cv=group_cv,
    groups=groups
)

print("GroupKFold Results:")
print(classification_report(subset['label_num'], y_pred_group))

GroupKFold Results:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     200.0
           1       0.00      0.00      0.00     200.0

    accuracy                           0.00     400.0
   macro avg       0.00      0.00      0.00     400.0
weighted avg       0.00      0.00      0.00     400.0

